# VIT Bot: LoRA fine-tune on Kaggle (Unsloth)

Before running, in the right-hand **Settings** panel:
- **Accelerator:** GPU T4 x2 (training uses one T4; the P100 is too old for current PyTorch)
- **Internet:** On (needs a phone-verified Kaggle account)
- **Input:** Add Input → Your Work / Datasets → `vit-bot-data` (holds `train.jsonl`, `val.jsonl`, `train.py`)

First time: run the cells one by one with `SMOKE = True` (about 5 min) to check everything works.
Then set `SMOKE = False` and click **Save Version → Save & Run All (Commit)**. That runs in the background,
so you can close the tab, and the outputs show up under the notebook's **Output** tab.

In [ ]:
!nvidia-smi

**1. Install Unsloth.** Kaggle already ships PyTorch + CUDA; this adds Unsloth, TRL, PEFT and bitsandbytes on top.

In [ ]:
%pip install -q unsloth

**2. Find the uploaded dataset.** Kaggle mounts inputs read-only under `/kaggle/input/`; anything we write goes to `/kaggle/working/` (kept as notebook output).

In [ ]:
import glob, os
hits = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
assert hits, "Dataset not attached: Add Input -> Your Datasets -> vit-bot-data"
DATA = os.path.dirname(hits[0])
print(DATA, os.listdir(DATA))
SMOKE = True  # True = 30-step test run; set False for the real training
# Qwen3.5 can't train in fp16 and the T4 has no bf16, so it falls back to slow fp32 here.
# Qwen3-4B-Instruct-2507 is plain attention and trains fine in fp16 on a T4.
MODEL = "unsloth/Qwen3-4B-Instruct-2507"

**3. Train.** The same `train.py` as on the laptop: 4-bit base model + LoRA adapters, loss only on the bot's replies.
Watch `loss` every 10 steps and `eval_loss` every 100: both should fall. If train loss keeps falling while eval loss rises, it's memorising.

In [ ]:
out = "/kaggle/working/smoke" if SMOKE else "/kaggle/working/vit-bot-lora"
cmd = f"python {DATA}/train.py --model {MODEL} --data-dir {DATA} --out {out}"
if SMOKE:
    cmd += " --max-steps 30"
print("Running:", cmd)
!{cmd}

**4. Try it: base vs fine-tuned.** One held-out prompt per mode. `disable_adapter()` temporarily switches the LoRA off, so the same loaded model gives both answers.

In [ ]:
import json
from unsloth import FastLanguageModel

OUT = "/kaggle/working/smoke/final" if SMOKE else "/kaggle/working/vit-bot-lora/final"
model, tok = FastLanguageModel.from_pretrained(OUT, max_seq_length=1024, load_in_4bit=True)
tok = getattr(tok, "tokenizer", tok)
FastLanguageModel.for_inference(model)

per_mode = {}
for line in open(f"{DATA}/val.jsonl"):
    msgs = json.loads(line)["messages"]
    per_mode.setdefault(msgs[0]["content"], msgs)

def reply(msgs):
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False,
                                  return_tensors="pt", return_dict=True).to("cuda")
    out = model.generate(**enc, max_new_tokens=200, do_sample=True, temperature=0.8, top_p=0.95)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

for system, msgs in per_mode.items():
    prompt = msgs[:2]  # system + first user message
    print("=" * 90)
    print(system[:70] + "...")
    print("USER:", msgs[1]["content"])
    with model.disable_adapter():
        print("\n--- BASE:\n" + reply(prompt))
    print("\n--- FINE-TUNED:\n" + reply(prompt))

**5. Package the adapter.** Download `vit-bot-lora.zip` from the Output tab (~100 MB), unzip into `outputs/` on the laptop. Merging + GGUF comes next.

In [ ]:
if not SMOKE:
    !cd /kaggle/working && zip -rq vit-bot-lora.zip vit-bot-lora/final vit-bot-lora/train_args.json && ls -lh vit-bot-lora.zip